In [0]:
%pip install yfinance

In [0]:
# Daily end-of-day price download -> MERGE into the existing bronze table.
#   * only the stocks in the "live universe" are refreshed (explicit list in conf/live.yml, or the N most traded);
#   * rows are matched on (symbol, calendar day): re-running is safe, overlapping days are corrected in place;
#   * the downloaded prices are compared with the stored ones on overlapping days; a systematic mismatch
#     (adjusted vs raw prices) stops the run instead of silently corrupting the history.
# Needs outbound internet access from the workspace. The first run bootstraps up to max_lookback_days
# of historical OHLCV directly from Yahoo Finance into Bronze Delta; later runs incrementally MERGE updates.
import sys
import os
sys.path.insert(0, os.getcwd())

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from live_data import (bar_cutoff_date, build_symbol_map, compute_start_date, normalize_download_frame,
                       price_basis_mismatch, utc_now)
from utils import load_project, load_yaml, table, widget_or
from datetime import timedelta

cfg = load_project()
live = load_yaml("live.yml")["live_data"]
d = cfg["data"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("symbols", "")                    # optional comma-separated override, e.g. AAPL,MSFT
dbutils.widgets.text("ignore_price_mismatch", "false")
# Set true ONLY for the one-time Yahoo Finance migration job. It replaces every row in
# the existing Bronze table after the Yahoo download succeeds; normal daily runs stay incremental.
dbutils.widgets.text("replace_existing_bronze", "false")
catalog = dbutils.widgets.get("catalog")
ignore_mismatch = dbutils.widgets.get("ignore_price_mismatch").strip().lower() in ("1", "true", "yes")
replace_existing_bronze = dbutils.widgets.get("replace_existing_bronze").strip().lower() in ("1", "true", "yes")

ts_col, sym_col = d["date_col"], d["symbol_col"]
col_map = {"symbol": sym_col, "timestamp": ts_col, "open": d["open_col"], "high": d["high_col"],
           "low": d["low_col"], "close": d["close_col"], "volume": d["volume_col"]}
if not sym_col:
    raise ValueError("Live ingestion needs a symbol column: set data.symbol_col in conf/project.yml.")
if live["provider"] != "yfinance":
    raise ValueError(f"Unsupported provider '{live['provider']}'. Only 'yfinance' is implemented.")

bronze_name = table(cfg, "bronze", catalog)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{cfg['tables']['bronze'].split('.')[0]}")
bronze_exists = spark.catalog.tableExists(bronze_name)
bronze = spark.table(bronze_name) if bronze_exists else None

# --------------------------------------------------------------------------------------
# 1. Which symbols to refresh
# --------------------------------------------------------------------------------------
override = [s.strip() for s in widget_or(dbutils.widgets.get("symbols"), "").split(",") if s.strip()]
universe = override or [str(s) for s in (live.get("symbols") or [])]
universe_table = f"{catalog}.{live['universe_table']}"

if not universe:
    if not bronze_exists:
        raise ValueError("Set at least one ticker in conf/live.yml live_data.symbols before the first Yahoo Finance bootstrap.")
    if spark.catalog.tableExists(universe_table):
        universe = [r["symbol"] for r in spark.table(universe_table).collect()]
        print(f"Using saved live universe ({len(universe)} symbols) from {universe_table}")
    else:
        newest = bronze.agg(F.max(ts_col)).collect()[0][0]
        recent = bronze.filter(F.col(ts_col) >= F.lit(newest) - F.expr(f"INTERVAL {int(live['universe_lookback_days'])} DAYS"))
        top = (recent.groupBy(sym_col)
                     .agg(F.avg(F.col(d["close_col"]).cast("double") * F.col(d["volume_col"]).cast("double")).alias("dollar_volume"),
                          F.count(F.lit(1)).alias("n_days"))
                     .filter(F.col("n_days") >= 20)
                     .orderBy(F.desc("dollar_volume"))
                     .limit(int(live["max_symbols"])))
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{live['universe_table'].split('.')[0]}")
        (top.select(F.col(sym_col).alias("symbol"), "dollar_volume")
            .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(universe_table))
        universe = [r["symbol"] for r in spark.table(universe_table).collect()]
        print(f"Selected the {len(universe)} most-traded symbols (saved to {universe_table}). "
              "Delete that table to re-select, or list symbols in conf/live.yml.")
print("Symbols:", universe[:15], "..." if len(universe) > 15 else "")

# --------------------------------------------------------------------------------------
# 2. Which dates to download
# --------------------------------------------------------------------------------------
now = utc_now()
today = now.date()
stored_last = ({r[sym_col]: r["last_ts"] for r in
                bronze.filter(F.col(sym_col).isin(universe)).groupBy(sym_col).agg(F.max(ts_col).alias("last_ts")).collect()}
               if bronze_exists else {})
last_dates = [stored_last[s].date() if s in stored_last and stored_last[s] is not None else None for s in universe]
# A replacement must fetch the full configured lookback, not just the overlap with old Parquet history.
if replace_existing_bronze:
    last_dates = [None] * len(universe)
start = compute_start_date(last_dates, live["overlap_days"], live["max_lookback_days"], today)
end_exclusive = today + timedelta(days=1)
print(f"Downloading daily bars {start} -> {today} (latest stored day per symbol: min={min([x for x in last_dates if x] or [None])})")

# --------------------------------------------------------------------------------------
# 3. Download (yfinance), tidy, drop a still-forming bar
# --------------------------------------------------------------------------------------
import yfinance as yf

symbol_map = build_symbol_map(universe)
tickers = list(symbol_map)
frames = []
try:
    for i in range(0, len(tickers), 100):
        chunk = tickers[i:i + 100]
        raw = yf.download(chunk, start=start.isoformat(), end=end_exclusive.isoformat(), interval="1d",
                          auto_adjust=bool(live["auto_adjust"]), group_by="ticker", threads=True, progress=False)
        frames.append(normalize_download_frame(raw, {t: symbol_map[t] for t in chunk}))
except Exception as e:
    raise RuntimeError(
        "Could not download live prices. Check workspace outbound internet access and Yahoo Finance availability. "
        f"Original error: {type(e).__name__}: {e}")

pdf = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(pdf):
    pdf = pdf[pdf["timestamp"].dt.date < bar_cutoff_date(now)]
got = set(pdf["symbol"]) if len(pdf) else set()
missing = [s for s in universe if s not in got]
print(f"Downloaded {len(pdf):,} bars for {len(got)}/{len(universe)} symbols. No data for: {missing[:20]}")
if len(pdf) == 0 or len(got) / len(universe) < float(live["min_symbol_success_ratio"]):
    raise RuntimeError("Too few symbols returned data (provider outage, rate limit, or wrong symbol names).")

# --------------------------------------------------------------------------------------
# 4. Shape into the bronze schema
# --------------------------------------------------------------------------------------
payload_cols = [sym_col, ts_col, d["open_col"], d["high_col"], d["low_col"], d["close_col"], d["volume_col"]]
if bronze_exists:
    target_types = {f.name: f.dataType for f in bronze.schema.fields}
    payload_cols = [c for c in bronze.columns if not c.startswith("_")]
else:
    target_types = {sym_col: "string", ts_col: "timestamp", d["open_col"]: "double",
                    d["high_col"]: "double", d["low_col"]: "double",
                    d["close_col"]: "double", d["volume_col"]: "double"}
sdf = spark.createDataFrame(pdf)
for src_name, dst_name in col_map.items():
    sdf = sdf.withColumnRenamed(src_name, dst_name)
for c in payload_cols:
    if c in sdf.columns:
        sdf = sdf.withColumn(c, F.col(c).cast(target_types[c]))
    else:                                              # extra columns present in the historical file
        sdf = sdf.withColumn(c, F.lit(None).cast(target_types[c]))
sdf = (sdf.select(*payload_cols)
          .withColumn("_row_hash", F.sha2(F.to_json(F.struct(*[F.col(c) for c in payload_cols])), 256))
          .withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_batch_id", F.lit("live-" + now.strftime("%Y%m%dT%H%M%S")))
          .withColumn("_source_file", F.lit("live:yfinance")))

# --------------------------------------------------------------------------------------
# 5. Safety check: do downloaded prices agree with stored prices on overlapping days?
# --------------------------------------------------------------------------------------
if bronze_exists:
    same_day = (F.col(f"s.`{sym_col}`") == F.col(f"t.`{sym_col}`")) & \
               (F.to_date(F.col(f"s.`{ts_col}`")) == F.to_date(F.col(f"t.`{ts_col}`")))
    dev = (sdf.alias("s").join(bronze.alias("t"), same_day)
              .select((F.col(f"s.`{d['close_col']}`").cast("double") / F.col(f"t.`{d['close_col']}`").cast("double") - 1).alias("dev"))
              .toPandas()["dev"])
    print(f"Overlapping stored days compared: {len(dev)}; median |deviation| = "
          f"{(dev.abs().median() if len(dev) else float('nan')):.5f}")
else:
    dev = pd.Series(dtype="float64")
    print("No Bronze table yet: bootstrapping historical data from Yahoo Finance.")
if (not replace_existing_bronze
        and price_basis_mismatch(dev, live["price_mismatch_tolerance"])
        and not ignore_mismatch):
    raise RuntimeError(
        "Downloaded prices differ from the stored history by more than "
        f"{live['price_mismatch_tolerance']:.0%} on overlapping days. The stored data is probably adjusted "
        "differently (split/dividend adjustment). Merging would create a price jump inside every series. "
        "Try auto_adjust: true in conf/live.yml, or re-run with ignore_price_mismatch=true if you accept it.")

# --------------------------------------------------------------------------------------
# 6. One-time replacement OR normal incremental MERGE
# --------------------------------------------------------------------------------------
if replace_existing_bronze:
    # The Yahoo download and schema validation have completed before this point.
    # Overwrite the SAME existing table, removing all old Parquet-derived rows and symbols
    # that are not part of the configured Yahoo Finance universe.
    print(f"REPLACEMENT MODE: overwriting existing table {bronze_name} with Yahoo Finance rows only.")
    (sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze_name))
    print("Replacement complete. Existing Parquet-derived Bronze rows have been replaced.")
else:
    # Daily mode: update the same symbol-day and insert new days; never truncate history.
    updates = {c: f"s.`{c}`" for c in payload_cols + ["_row_hash", "_ingest_ts", "_batch_id", "_source_file"]}
    if bronze_exists:
        (DeltaTable.forName(spark, bronze_name).alias("t")
            .merge(sdf.alias("s"), f"t.`{sym_col}` = s.`{sym_col}` AND to_date(t.`{ts_col}`) = to_date(s.`{ts_col}`)")
            .whenMatchedUpdate(set=updates)
            .whenNotMatchedInsertAll()
            .execute())
    else:
        sdf.write.format("delta").mode("overwrite").saveAsTable(bronze_name)

newest_day = pdf["timestamp"].max().date()
print(f"Live ingestion done ({'REPLACE' if replace_existing_bronze else 'INCREMENTAL MERGE'}). "
      f"Newest day in bronze for the live universe: {newest_day}. "
      f"Total bronze rows: {spark.table(bronze_name).count():,}")
source_counts = (spark.table(bronze_name).groupBy("_source_file").count().orderBy(F.desc("count")).collect()
                if "_source_file" in spark.table(bronze_name).columns else [])
print("Bronze source counts:", [(r["_source_file"], r["count"]) for r in source_counts])
dbutils.jobs.taskValues.set(key="newest_day", value=str(newest_day))
dbutils.jobs.taskValues.set(key="rows_downloaded", value=int(len(pdf)))
dbutils.jobs.taskValues.set(key="symbols_refreshed", value=int(len(got)))